# Hybrid NER + Classification Pipeline for OJT Journal Task Tagging
### Deterministic Dictionary + Contextual Transformer NER

This notebook provides an end-to-end, reproducible workflow for the OJT Task Extraction system:
1. **Environment Setup & GPU Acceleration**: PyTorch 2.6.0+cu124, spaCy 3.8.16, transformers.
2. **Real Annotated Data Ingestion**: Cleaned student journal entries (`data/data.jsonl`).
3. **Seed Terms Dictionary**: Seed lookup (`data/terms.csv`) normalized to `IT_TERM` / `CLERICAL_TERM`.
4. **Leak-Free Real Dataset Partitioning**: Sentence-level connected-component splitting (`train.spacy`, `dev.spacy`, `test.spacy`).
5. **7-Check Data Leakage Audit**: Strict validation across splits, unseen benchmark terms, and synthetic near-duplicates (similarity $\ge 0.70$).
6. **TRTR (Train Real, Test Real)**: Baseline — transformer fine-tuned on real data only.
7. **TRSTR via Paraphrasing**: Augmentation using T5 paraphraser (`Vamsi/T5_Paraphrase_Paws`).
8. **TRSTR via LLM Generation**: Augmentation using Gemini LLM-direct generation (`scripts/generate_llm_synthetic.py`).
9. **TRSTR-LLM Mini**: Lightweight `distilroberta-base` alternative trained on the same augmented pool.
10. **TRSTR-LLM CNN**: Ultra-lightweight tok2vec CNN alternative for highly constrained hardware.
11. **Side-by-Side Comparative Evaluation**: Main conditions on the same held-out test set and unseen benchmark.


---
## Phase 1: Environment Setup & GPU Initialization

Import core modules and initialize GPU acceleration.

In [1]:
import os
import sys
import json
import re
import random
import time
import shutil
import pandas as pd
import spacy
from spacy.tokens import DocBin

# Add project root to sys.path
sys.path.insert(0, os.path.abspath("."))

import scripts
from scripts.annotation import (
    load_terms_dictionary,
    find_term_spans,
    load_jsonl,
    report_dataset_diagnostics,
    prepare_real_data_pipeline,
    convert_records_to_docbin,
)
from scripts.training import train_ner_trf
from scripts.pipeline import HybridJournalPipeline
from scripts.eval import (
    evaluate_unseen_generalization,
    evaluate_test_docbin,
    evaluate_real_holdout,
    load_unseen_benchmark,
)
from scripts.check_data_leakage import run_all_leakage_checks

# Initialize GPU acceleration
gpu_ready = scripts.init_gpu()
print(f"System Status: GPU Acceleration Active = {gpu_ready}")

[2026-11-07 05:28:35,947] INFO: GPU accelerated with NVIDIA GeForce RTX 3060 Laptop GPU (spacy.require_gpu=True)


System Status: GPU Acceleration Active = True


---
## Phase 2: Load & Inspect Real Annotated Data

Load the authentic, manually annotated OJT student journal dataset (`data/data.jsonl`) and audit its composition diagnostics (negative ratio target: 25–35%, class balance ratio $\le 1.5:1$).

In [2]:
data_path = "data/data.jsonl"
records = load_jsonl(data_path)
print(f"Loaded {len(records)} records from {data_path}\n")

# Report dataset diagnostics (negative ratio, class balance)
diagnostics = report_dataset_diagnostics(records, dataset_label="Real Annotated Dataset")

# Display sample records with highlighted entity spans
print("\nSample Records:")
for r in records[:5]:
    text = r["text"]
    ents = [(text[e["start"]:e["end"]], e["label"]) for e in r.get("entities", [])]
    print(f"  Text    : {text}")
    print(f"  Entities: {ents if ents else '(Negative — 0 entities)'}\n")

[2026-11-07 05:28:35,963] INFO: --- Real Annotated Dataset Diagnostics ---
[2026-11-07 05:28:35,963] INFO:   Total records : 1320
[2026-11-07 05:28:35,964] INFO:   Positives     : 868
[2026-11-07 05:28:35,964] INFO:   Negatives     : 452 (34.2%)
[2026-11-07 05:28:35,964] INFO:   Entity labels : {'CLERICAL_TERM': 541, 'IT_TERM': 933}
[2026-11-07 05:28:35,965] INFO:   Negative ratio is within target range (25-35%).
[2026-11-07 05:28:35,965] INFO:   Class balance acceptable: IT_TERM=933, CLERICAL_TERM=541 (ratio 1.72:1)


Loaded 1320 records from data/data.jsonl


Sample Records:
  Text    : Encoded and organized vendor evaluation data using Microsoft Excel.
  Entities: [('Encoded', 'CLERICAL_TERM'), ('vendor evaluation data', 'CLERICAL_TERM'), ('Microsoft Excel', 'CLERICAL_TERM')]

  Text    : Organized and digitized vendor-related documents.
  Entities: [('vendor-related documents', 'CLERICAL_TERM')]

  Text    : Created vendor profiles in the system/database.
  Entities: [('vendor profiles', 'CLERICAL_TERM'), ('database', 'IT_TERM')]

  Text    : Designed identification (ID) cards for vendors/staff.
  Entities: [('identification (ID) cards', 'CLERICAL_TERM')]

  Text    : Managed and updated records in the office database.
  Entities: [('office database', 'IT_TERM')]



---
## Phase 3: Seed Terms Dictionary & Weak Supervision

Inspect `data/terms.csv` containing normalized entity seeds (`IT_TERM` and `CLERICAL_TERM`).

In [3]:
terms_df = pd.read_csv("data/terms.csv")
terms_dict = load_terms_dictionary("data/terms.csv")

print(f"Terms Dictionary: {len(terms_dict)} unique terms loaded.")
print(f"Label distribution:")
print(terms_df["label"].value_counts())

[2026-11-07 05:28:35,975] INFO: Loaded 372 unique terms from data/terms.csv


Terms Dictionary: 372 unique terms loaded.
Label distribution:
label
IT_TERM          215
CLERICAL_TERM    157
Name: count, dtype: int64


---
## Phase 4: Deduplication & Train/Dev/Test Split (Pure Real Data)

Split `data/data.jsonl` into 70% train, 15% dev, and 15% test partitions using sentence-level connected-component grouping to guarantee **zero sentence leakage** across splits.

In [4]:
pipeline_result = prepare_real_data_pipeline(
    data_jsonl_path="data/data.jsonl",
    output_dir="data/training",
    train_ratio=0.70,
    dev_ratio=0.15,
    seed=42
)

print("\nDocBin files written:")
for split_name, path in pipeline_result["spacy_paths"].items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"  {split_name:<5}: {path} ({size} bytes)")

[2026-11-07 05:28:35,991] INFO: Loaded 1320 records from data/data.jsonl
[2026-11-07 05:28:35,992] INFO: --- Full Dataset Diagnostics ---
[2026-11-07 05:28:35,993] INFO:   Total records : 1320
[2026-11-07 05:28:35,993] INFO:   Positives     : 868
[2026-11-07 05:28:35,993] INFO:   Negatives     : 452 (34.2%)
[2026-11-07 05:28:35,993] INFO:   Entity labels : {'CLERICAL_TERM': 541, 'IT_TERM': 933}
[2026-11-07 05:28:35,994] INFO:   Negative ratio is within target range (25-35%).
[2026-11-07 05:28:35,994] INFO:   Class balance acceptable: IT_TERM=933, CLERICAL_TERM=541 (ratio 1.72:1)
[2026-11-07 05:28:35,998] INFO: Deduplication complete: 1320 -> 1320 records (0 duplicates removed, 0 annotation conflicts).
/home/caineirb/Documents/PauPau/spaCy-training/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2


DocBin files written:
  train: data/training/train.spacy (142150 bytes)
  dev  : data/training/dev.spacy (36266 bytes)
  test : data/training/test.spacy (34004 bytes)


---
## Phase 5: 7-Check Data Leakage & Benchmark Isolation Audit

Verify strict zero data leakage across all 7 checks before evaluating models:
1. Cross-Split Document Duplication = 0
2. Cross-Split Sentence Duplication = 0
3. Terms CSV Unseen Benchmark Leakage = 0
4. Training Annotation Unseen Benchmark Leakage = 0
5. EntityRuler Matchability on Unseen Benchmark = 0
6. Real Holdout Data Isolation = 0
7. **Synthetic Pool Isolation**: 0 exact or near-duplicates (similarity $\ge 0.70$) between synthetic pool and evaluation partitions; 0 benchmark terms in synthetic pool.

In [5]:
all_passed = run_all_leakage_checks()
print(f"\nOverall Leakage Audit Result: {'ALL CHECKS PASSED (ZERO LEAKAGE)' if all_passed else 'LEAKAGE DETECTED'}")

[2026-11-07 05:28:38,837] INFO: Loaded 85 unseen benchmark samples from data/test/unseen_benchmark.jsonl
[2026-11-07 05:28:38,838] INFO: Loaded 372 unique terms from data/terms.csv
[2026-11-07 05:28:38,839] INFO: Loading transformer model from 'models/ner_trf/model-best'...



###########################################################################
  DATA LEAKAGE & BENCHMARK ISOLATION AUDIT
###########################################################################

[INFO] Loading dataset splits from data/training/*.spacy...
[INFO] Initializing pipeline for EntityRuler matchability audit...


[2026-11-07 05:28:42,321] INFO: Configured standalone Dictionary matcher with 372 patterns.
[2026-11-07 05:28:42,395] INFO: Loaded 372 unique terms from data/terms.csv



  Check 1: Cross-Split Document Duplication
[PASS] Zero duplicate documents across train/dev/test splits.
       Docs: train=923, dev=198, test=199

  Check 2: Cross-Split Sentence Duplication
[PASS] Zero duplicate sentences across train/dev/test splits.
       Sentences: train=1122, dev=242, test=233

  Check 3: Unseen Benchmark Terms in data/terms.csv
[PASS] Zero of 65 unseen benchmark terms appear in data/terms.csv.
       Dictionary size checked: 372 terms.

  Check 4: Unseen Benchmark Terms in Training Annotations
[PASS] Zero of 65 unseen benchmark terms appear in data/data.jsonl.
       Annotated entity pool: 713 unique entity strings.

  Check 5: EntityRuler Matchability on Unseen Benchmark
[PASS] Zero unseen benchmark terms are matchable as exact entities by the EntityRuler.

  Check 6: Real Holdout Scaffolding & Data Isolation
[PASS] Real holdout dataset is currently empty (0 sentences; awaiting manual gold annotations).
       Scanned holdout.jsonl and data/test/raw/ against

---
---
# CONDITION 1: TRTR (Train Real, Test Real)
---
## Phase 6: Transformer Fine-Tuning (TRTR)

Fine-tune the RoBERTa-base transformer NER model on the **real training partition only** with patience-based early stopping:
- `max_steps = 2500`
- `eval_frequency = 50`
- `patience = 400`
- `use_gpu = 0`

In [6]:
# Free GPU memory and reload training module
import gc, torch, importlib
import scripts.training
importlib.reload(scripts.training)
from scripts.training import train_ner_trf

for var_name in ["pipe_trtr", "pipe_trstr_para", "pipe_trstr_llm"]:
    if var_name in globals(): del globals()[var_name]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

# Train TRTR model from scratch
trtr_model_dir = "models/ner_trf_trtr"
trtr_best = os.path.join(trtr_model_dir, "model-best")

# Clean previous model to train from scratch
if os.path.exists(trtr_model_dir):
    shutil.rmtree(trtr_model_dir)
    print(f"Cleaned previous TRTR model at {trtr_model_dir}")

print("Fine-tuning TRTR model on pure real data (data/training/train.spacy)...")
train_ner_trf(
    output_dir=trtr_model_dir,
    train_path="data/training/train.spacy",
    dev_path="data/training/dev.spacy",
    config_path="config_trf.cfg",
    use_gpu=0,
    max_steps=2500,
    eval_frequency=50,
    patience=400
)
print(f"TRTR model trained. Best checkpoint: {trtr_best}")


[2026-11-07 05:28:44,739] INFO: Starting transformer training on GPU 0 for up to 2500 steps (eval_frequency=50, patience=400 steps, max_batch_items=2048)...
[2026-11-07 05:28:44,842] [INFO] Set up nlp object from config
[2026-11-07 05:28:44,842] INFO: Set up nlp object from config
[2026-11-07 05:28:44,853] [INFO] Pipeline: ['transformer', 'ner']
[2026-11-07 05:28:44,853] INFO: Pipeline: ['transformer', 'ner']
[2026-11-07 05:28:44,858] [INFO] Created vocabulary
[2026-11-07 05:28:44,858] INFO: Created vocabulary
[2026-11-07 05:28:44,858] [INFO] Finished initializing nlp object
[2026-11-07 05:28:44,858] INFO: Finished initializing nlp object


Cleaned previous TRTR model at models/ner_trf_trtr
Fine-tuning TRTR model on pure real data (data/training/train.spacy)...
ℹ Saving to output directory: models/ner_trf_trtr
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[2026-11-07 05:28:49,167] [INFO] Initialized pipeline components: ['transformer', 'ner']
[2026-11-07 05:28:49,167] INFO: Initialized pipeline components: ['transformer', 'ner']


✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  -------------  --------  ------  ------  ------  ------
  0       0        7326.39    289.65    0.00    0.00    0.00    0.00
  3      50      100652.18  27427.65    0.00    0.00    0.00    0.00
  6     100        4306.74   5696.75   27.56   52.00   18.75    0.28
 10     150        3754.88   4606.61   41.61   37.26   47.12    0.42
 13     200        3113.66   3822.61   47.43   48.26   46.63    0.47
 16     250        2566.22   3108.50   55.70   62.13   50.48    0.56
 20     300        1937.79   2396.08   60.55   57.89   63.46    0.61
 23     350        1574.34   2121.91   66.98   65.74   68.27    0.67
 26     400        1330.24   1787.14   61.24   60.95   61.54    0.61
 30     450        1039.62   1527.88   64.44   63.98   64.90    0.64
 33     5

[2026-11-07 05:36:12,174] INFO: Training completed. Best model checkpoint: models/ner_trf_trtr/model-best


✔ Saved pipeline to output directory
models/ner_trf_trtr/model-last
TRTR model trained. Best checkpoint: models/ner_trf_trtr/model-best


---
## Phase 7: Evaluation (TRTR Baseline)

Evaluate the TRTR model on the real evaluation partitions:
1. **Held-Out Real Test Set** (`data/training/test.spacy`)
2. **Unseen-Term Benchmark** (`data/test/unseen_benchmark.jsonl` — 65 unseen enterprise technologies)

In [7]:
# Initialize TRTR pipeline
pipe_trtr = HybridJournalPipeline(model_path=trtr_best, terms_csv_path="data/terms.csv")

# 1. Evaluate on Real Held-Out Test Set
test_trtr = evaluate_test_docbin(pipe_trtr, "data/training/test.spacy", run_label="trtr")

# 2. Evaluate on Unseen Benchmark (all three modes)
unseen_trtr = evaluate_unseen_generalization(pipe_trtr, run_label="trtr")

# Print TRTR Results
print("                       TRTR (Real Only) BASELINE RESULTS")
print("=" * 75)
print(f"{'METRIC':<42} | {'TRTR (Real Only)':>18}")
print("-" * 75)
print(f"{'Held-Out Real Test Overall F1':<42} | {test_trtr['overall_f1']:>16}%")
print(f"{'Held-Out Real Test Overall Precision':<42} | {test_trtr['overall_precision']:>16}%")
print(f"{'Held-Out Real Test Overall Recall':<42} | {test_trtr['overall_recall']:>16}%")
print("-" * 75)
for lbl in ['IT_TERM', 'CLERICAL_TERM']:
    sl = test_trtr['labels'].get(lbl, {})
    print(f"  {lbl + ' F1':<40} | {sl.get('f1', 0):>16}%")
    print(f"  {lbl + ' Precision':<40} | {sl.get('precision', 0):>16}%")
    print(f"  {lbl + ' Recall':<40} | {sl.get('recall', 0):>16}%")
print("-" * 75)

b_trf = unseen_trtr['modes']['transformer_only']
print(f"{'Unseen Benchmark Transformer Recall':<42} | {b_trf.get('recall_pct', 0):>16}%")
print(f"{'Unseen Benchmark Transformer Precision':<42} | {b_trf.get('precision_pct', 0):>16}%")
print(f"{'Unseen Benchmark Transformer F1':<42} | {b_trf.get('f1_pct', 0):>16}%")
b_hyb = unseen_trtr['modes']['hybrid']
print(f"{'Unseen Benchmark Hybrid Recall':<42} | {b_hyb.get('recall_pct', 0):>16}%")
print(f"{'Unseen Benchmark Hybrid Precision':<42} | {b_hyb.get('precision_pct', 0):>16}%")
print(f"{'Unseen Benchmark Hybrid F1':<42} | {b_hyb.get('f1_pct', 0):>16}%")
print(f"{'Unseen Benchmark (Generalization Lift)':<42} | {unseen_trtr.get('generalization_lift', 'N/A'):>17}")
print("=" * 75)

[2026-11-07 05:36:12,197] INFO: Loaded 372 unique terms from data/terms.csv
[2026-11-07 05:36:12,198] INFO: Loading transformer model from 'models/ner_trf_trtr/model-best'...
[2026-11-07 05:36:15,532] INFO: Configured standalone Dictionary matcher with 372 patterns.
[2026-11-07 05:36:16,266] INFO: [DICTIONARY OVERRIDE] Span 'automated Excel tool' (14:34): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 05:36:18,924] INFO: [DICTIONARY OVERRIDE] Span 'office database' (38:53): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 05:36:20,242] INFO: [DICTIONARY OVERRIDE] Span 'student records' (35:50): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 05:36:22,674] INFO: [DICTIONARY OVERRIDE] Span 'database' (43:51): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 05:36:23,808] INFO: [DICTIONARY OVERRIDE] Span 'Record management systems' (0:25): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 05:36:25,212] INFO: Per-item results: d

                       TRTR (Real Only) BASELINE RESULTS
METRIC                                     |   TRTR (Real Only)
---------------------------------------------------------------------------
Held-Out Real Test Overall F1              |            71.27%
Held-Out Real Test Overall Precision       |            71.74%
Held-Out Real Test Overall Recall          |            70.82%
---------------------------------------------------------------------------
  IT_TERM F1                               |            71.52%
  IT_TERM Precision                        |             72.9%
  IT_TERM Recall                           |            70.19%
  CLERICAL_TERM F1                         |            70.75%
  CLERICAL_TERM Precision                  |            69.33%
  CLERICAL_TERM Recall                     |            72.22%
---------------------------------------------------------------------------
Unseen Benchmark Transformer Recall        |            78.46%
Unseen Benchmark Tran

---
---
# CONDITION 2: TRSTR via Paraphrasing (T5)
---
## Phase 8: Synthetic Paraphrase Generation

Generate paraphrase-based synthetic augmentation using `Vamsi/T5_Paraphrase_Paws`.

Key design:
- Paraphrase only the **real training split** sentences (never dev/test)
- Entity text must be preserved literally (exact substring match with word-boundary validation)
- Negative sentences are also paraphrased (preserving zero-entity status)
- Generic noun blocklist filters applied
- Start/end offsets re-computed via exact string search

In [8]:
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load T5 paraphraser
PARAPHRASE_MODEL_NAME = "Vamsi/T5_Paraphrase_Paws"
print(f"Loading paraphraser: {PARAPHRASE_MODEL_NAME}...")
para_tokenizer = T5Tokenizer.from_pretrained(PARAPHRASE_MODEL_NAME)
para_model = T5ForConditionalGeneration.from_pretrained(PARAPHRASE_MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
para_model = para_model.to(device)
para_model.eval()
print(f"Paraphraser loaded on {device}.")

Loading paraphraser: Vamsi/T5_Paraphrase_Paws...


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Paraphraser loaded on cuda.


In [9]:
def generate_paraphrases(text, model, tokenizer, num_return_sequences=5, max_length=128):
    """Generate paraphrase candidates using the T5 model."""
    input_text = f"paraphrase: {text} </s>"
    encoding = tokenizer.encode_plus(
        input_text,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=max_length,
            num_beams=10,
            num_return_sequences=num_return_sequences,
            no_repeat_ngram_size=2,
            early_stopping=True,
        )

    candidates = []
    for output in outputs:
        decoded = tokenizer.decode(output, skip_special_tokens=True)
        decoded = decoded.strip()
        if decoded and decoded.lower() != text.lower():
            candidates.append(decoded)
    return candidates


def relocate_entities(original_text, paraphrase_text, original_entities):
    """Relocate entity spans from original to paraphrase via exact substring match.
    
    Returns (success, relocated_entities) where success=False if any entity
    text cannot be found in the paraphrase with valid word boundaries.
    """
    relocated = []
    used_ranges = []
    
    for ent in original_entities:
        entity_text = original_text[ent["start"]:ent["end"]]
        label = ent["label"]
        
        if not entity_text.strip():
            return False, []
        
        # Search for exact substring match with word-boundary validation
        search_start = 0
        found = False
        while search_start < len(paraphrase_text):
            # Case-sensitive first, then case-insensitive fallback
            pos = paraphrase_text.find(entity_text, search_start)
            if pos == -1:
                # Try case-insensitive
                pos = paraphrase_text.lower().find(entity_text.lower(), search_start)
                if pos == -1:
                    break
            
            start_idx = pos
            end_idx = pos + len(entity_text)
            
            # Word-boundary check
            before = paraphrase_text[start_idx - 1] if start_idx > 0 else " "
            after = paraphrase_text[end_idx] if end_idx < len(paraphrase_text) else " "
            
            if before.isalnum() or after.isalnum():
                search_start = pos + 1
                continue
            
            # Check overlap
            overlap = any(max(start_idx, us) < min(end_idx, ue) for us, ue in used_ranges)
            if overlap:
                search_start = pos + 1
                continue
            
            # Use the actual text from the paraphrase (preserving case)
            actual_text = paraphrase_text[start_idx:end_idx]
            relocated.append({"start": start_idx, "end": end_idx, "label": label})
            used_ranges.append((start_idx, end_idx))
            found = True
            break
        
        if not found:
            return False, []
    
    # Sort by start offset and verify no overlaps
    relocated.sort(key=lambda e: e["start"])
    for i in range(len(relocated) - 1):
        if relocated[i]["end"] > relocated[i + 1]["start"]:
            return False, []
    
    return True, relocated


# Generic noun blocklist (same as in generate_llm_synthetic.py)
GENERIC_NOUN_BLOCKLIST = {
    "program", "encode", "encoded", "office documents",
    "coordination", "orientation", "deployment", "front page",
    "copies", "issued", "stalls", "proctoring",
    "reference numbers", "field trials", "data quality",
    "system workflows", "data requirements", "system exploration",
    # Rejected generic administrative artifacts & paperwork (Group 3 policy decision)
    "visitor ids", "visitor id", "graduation credentials", "graduation credential",
    "faculty academic records", "faculty academic record", "office supply lists", "office supply list",
    "court summons", "employee leave records", "employee leave record", "leave records",
    "fiscal department line items", "fiscal department line item", "payroll discrepancies", "payroll discrepancy",
    "deed logs", "deed log", "public oaths", "public oath", "financial aid allowances", "financial aid allowance",
    "employee tax forms", "employee tax form", "tax forms", "warehouse receipt ledgers", "warehouse receipt ledger",
    # Rejected non-task environmental & conversational artifacts (Group 4 policy decision)
    "blockers", "blocker", "attendance machine", "intern time log sheet", "time log sheet",
    "visitor logbook", "guest identification badges", "guest identification badge",
    "notebook notes", "notebook note", "projector monitors", "projector monitor",
}

BARE_GENERICS = {
    "system", "systems", "database", "databases", "software", "softwares",
    "code", "codes", "computer", "computers", "website", "websites",
    "application", "applications", "server", "servers", "backend", "backends",
    "frontend", "frontends", "data", "table", "tables", "document", "documents",
    "file", "files", "record", "records", "form", "forms", "paper", "papers",
    "program", "programs", "network", "networks", "printer", "printers",
    "scanner", "scanners", "monitor", "monitors", "keyboard", "keyboards",
    "desk", "desks", "office", "offices", "university", "universities",
    "folder", "folders", "department", "departments",
}


def passes_paraphrase_quality_check(text, entities, seen_texts, eval_sentences):
    """Quality check for a paraphrase candidate."""
    if not text or len(text.strip()) < 10 or len(text) > 500:
        return False, "Length"
    
    text_lower = text.strip().lower()
    if text_lower in seen_texts:
        return False, "Duplicate"
    if text_lower in eval_sentences:
        return False, "Eval leak"
    
    for ent in entities:
        term = text[ent["start"]:ent["end"]].strip().lower()
        if term in GENERIC_NOUN_BLOCKLIST or term in BARE_GENERICS:
            return False, f"Blocked term: {term}"
    
    return True, "OK"


print("Paraphrase generation utilities defined.")

Paraphrase generation utilities defined.


In [10]:
# Load the real training split records for paraphrasing
nlp_blank = spacy.blank("en")
train_db = DocBin().from_disk("data/training/train.spacy")
train_docs = list(train_db.get_docs(nlp_blank.vocab))

train_records = []
for doc in train_docs:
    train_records.append({
        "text": doc.text,
        "entities": [
            {"start": ent.start_char, "end": ent.end_char, "label": ent.label_}
            for ent in doc.ents
        ],
    })

print(f"Loaded {len(train_records)} real training records for paraphrasing.")
print(f"  Positive: {sum(1 for r in train_records if r['entities'])}")
print(f"  Negative: {sum(1 for r in train_records if not r['entities'])}")

# Build evaluation sentence set for dedup guard
eval_sentences = set()
for split in ["dev", "test"]:
    path = f"data/training/{split}.spacy"
    if os.path.exists(path):
        db = DocBin().from_disk(path)
        for doc in db.get_docs(nlp_blank.vocab):
            eval_sentences.add(doc.text.strip().lower())

for eval_path in ["data/test/unseen_benchmark.jsonl", "data/test/holdout.jsonl"]:
    if os.path.exists(eval_path):
        with open(eval_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        rec = json.loads(line)
                        eval_sentences.add(rec["text"].strip().lower())
                    except Exception:
                        pass

# Include all real training sentences too
real_sentences = {r["text"].strip().lower() for r in train_records}
eval_sentences.update(real_sentences)

print(f"Dedup guard: {len(eval_sentences)} protected sentences.")

Loaded 923 real training records for paraphrasing.
  Positive: 605
  Negative: 318
Dedup guard: 1405 protected sentences.


In [11]:
# Generate paraphrases via standalone script to ensure reproducibility and prevent logic drift
import subprocess, sys
print("Running standalone paraphrase generation script (scripts/generate_paraphrases.py)...")
subprocess.run([sys.executable, "scripts/generate_paraphrases.py", "--seed", "42"], check=True)
paraphrase_records = load_jsonl("data/synthetic_paraphrases.jsonl")
print(f"Loaded {len(paraphrase_records)} paraphrases with full source provenance from data/synthetic_paraphrases.jsonl")


Running standalone paraphrase generation script (scripts/generate_paraphrases.py)...


[2026-11-07 05:36:38,448] INFO: GPU accelerated with NVIDIA GeForce RTX 3060 Laptop GPU (spacy.require_gpu=True)


Loading training records from data/training/train.spacy...
Loaded 923 real training records for paraphrasing.
  Positive: 605
  Negative: 318
Dedup guard: 1405 protected sentences.

Loading paraphraser: Vamsi/T5_Paraphrase_Paws...


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Paraphraser loaded on cuda.

Generating paraphrases for 923 training records...
Target: ~1 accepted paraphrase per record (best-of-5 candidates)



/home/caineirb/Documents/PauPau/spaCy-training/.venv/lib/python3.11/site-packages/transformers/models/t5/tokenization_t5.py:289: UserWarning: This sequence already has </s>. In future versions this behavior may lead to duplicated eos tokens being added.
  warnings.warn(


  Progress: 100/923 records processed, 64 paraphrases accepted
  Progress: 200/923 records processed, 124 paraphrases accepted
  Progress: 300/923 records processed, 185 paraphrases accepted
  Progress: 400/923 records processed, 249 paraphrases accepted
  Progress: 500/923 records processed, 316 paraphrases accepted
  Progress: 600/923 records processed, 379 paraphrases accepted
  Progress: 700/923 records processed, 437 paraphrases accepted
  Progress: 800/923 records processed, 494 paraphrases accepted
  Progress: 900/923 records processed, 551 paraphrases accepted

PARAPHRASE GENERATION COMPLETE
Total accepted paraphrases : 565
  Positive paraphrases     : 250
  Negative paraphrases     : 315 (55.8%)
  Span relocation failures : 1347
  Quality check failures   : 496
  Total candidates tried   : 4569

Saved 565 paraphrases to data/synthetic_paraphrases.jsonl
Unique source IDs: 565 (from 923 source records)
Loaded 565 paraphrases with full source provenance from data/synthetic_paraph

In [12]:
# Free GPU memory from T5 model before training
del para_model, para_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("T5 paraphraser unloaded from GPU memory.")

T5 paraphraser unloaded from GPU memory.


---
## Phase 9: Build TRSTR-Paraphrase Training Dataset & Train

Combine real training records with accepted paraphrases into a single training DocBin, then fine-tune a fresh transformer.

In [13]:
# Build combined TRSTR-Paraphrase training dataset
from scripts.annotation import convert_records_to_docbin

# Load paraphrases (in case you re-run from here)
if not paraphrase_records:
    paraphrase_records = load_jsonl(PARAPHRASE_OUTPUT_PATH)

# Combine real + paraphrase
combined_para_records = []
for r in train_records:
    combined_para_records.append({**r, "augmentation_type": "real"})
for r in paraphrase_records:
    combined_para_records.append(r)

print(f"TRSTR-Paraphrase combined training pool: {len(combined_para_records)} records")
print(f"  Real     : {len(train_records)}")
print(f"  Synthetic: {len(paraphrase_records)}")

# Report diagnostics
report_dataset_diagnostics(combined_para_records, dataset_label="TRSTR-Paraphrase Combined Pool")

# Save combined JSONL
TRSTR_PARA_JSONL = "data/training_trstr_paraphrase.jsonl"
with open(TRSTR_PARA_JSONL, "w", encoding="utf-8") as f:
    for rec in combined_para_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"Saved combined JSONL to {TRSTR_PARA_JSONL}")

# Convert to DocBin
TRSTR_PARA_SPACY = "data/training/train_trstr_paraphrase.spacy"
convert_records_to_docbin(combined_para_records, TRSTR_PARA_SPACY)
print(f"Saved DocBin to {TRSTR_PARA_SPACY} ({os.path.getsize(TRSTR_PARA_SPACY)} bytes)")

[2026-11-07 05:44:22,081] INFO: --- TRSTR-Paraphrase Combined Pool Diagnostics ---
[2026-11-07 05:44:22,082] INFO:   Total records : 1488
[2026-11-07 05:44:22,083] INFO:   Positives     : 855
[2026-11-07 05:44:22,083] INFO:   Negatives     : 633 (42.5%)
[2026-11-07 05:44:22,084] INFO:   Entity labels : {'IT_TERM': 871, 'CLERICAL_TERM': 545}
[2026-11-07 05:44:22,085] WARNING:   Negative ratio (42.5%) is above target range (25-35%). Consider adding more positive examples.
[2026-11-07 05:44:22,086] INFO:   Class balance acceptable: IT_TERM=871, CLERICAL_TERM=545 (ratio 1.6:1)


TRSTR-Paraphrase combined training pool: 1488 records
  Real     : 923
  Synthetic: 565
Saved combined JSONL to data/training_trstr_paraphrase.jsonl


[2026-11-07 05:44:22,468] INFO: Saved 1488 docs (1416 valid entities, 0 dropped) to data/training/train_trstr_paraphrase.spacy


Saved DocBin to data/training/train_trstr_paraphrase.spacy (222039 bytes)


In [14]:
# Free GPU memory and reload training module
import gc, torch, importlib
import scripts.training
importlib.reload(scripts.training)
from scripts.training import train_ner_trf

for var_name in ["pipe_trtr", "pipe_trstr_para", "pipe_trstr_llm", "para_model", "para_tokenizer"]:
    if var_name in globals(): del globals()[var_name]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

# Train TRSTR-Paraphrase model from scratch
trstr_para_model_dir = "models/ner_trf_trstr_paraphrase"
trstr_para_best = os.path.join(trstr_para_model_dir, "model-best")

# Clean previous model to train from scratch
if os.path.exists(trstr_para_model_dir):
    shutil.rmtree(trstr_para_model_dir)
    print(f"Cleaned previous TRSTR-Paraphrase model at {trstr_para_model_dir}")

print("Fine-tuning TRSTR-Paraphrase model...")
train_ner_trf(
    output_dir=trstr_para_model_dir,
    train_path=TRSTR_PARA_SPACY,
    dev_path="data/training/dev.spacy",
    config_path="config_trf.cfg",
    use_gpu=0,
    max_steps=2500,
    eval_frequency=50,
    patience=400
)
print(f"TRSTR-Paraphrase model trained. Best checkpoint: {trstr_para_best}")


[2026-11-07 05:44:23,845] INFO: Starting transformer training on GPU 0 for up to 2500 steps (eval_frequency=50, patience=400 steps, max_batch_items=2048)...
[2026-11-07 05:44:23,962] [INFO] Set up nlp object from config
[2026-11-07 05:44:23,962] INFO: Set up nlp object from config
[2026-11-07 05:44:23,976] [INFO] Pipeline: ['transformer', 'ner']
[2026-11-07 05:44:23,976] INFO: Pipeline: ['transformer', 'ner']
[2026-11-07 05:44:23,981] [INFO] Created vocabulary
[2026-11-07 05:44:23,981] INFO: Created vocabulary
[2026-11-07 05:44:23,982] [INFO] Finished initializing nlp object
[2026-11-07 05:44:23,982] INFO: Finished initializing nlp object


Cleaned previous TRSTR-Paraphrase model at models/ner_trf_trstr_paraphrase
Fine-tuning TRSTR-Paraphrase model...
ℹ Saving to output directory: models/ner_trf_trstr_paraphrase
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[2026-11-07 05:44:25,946] [INFO] Initialized pipeline components: ['transformer', 'ner']
[2026-11-07 05:44:25,946] INFO: Initialized pipeline components: ['transformer', 'ner']


✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  -------------  --------  ------  ------  ------  ------
  0       0        3593.41    351.42    0.00    0.00    0.00    0.00
  2      50       98596.79  28269.25    0.00    0.00    0.00    0.00
  4     100        3559.19   4981.31   28.28   50.00   19.71    0.28
  6     150        3121.15   3819.77   43.93   47.49   40.87    0.44
  8     200        2574.49   3065.96   55.07   48.36   63.94    0.55
 10     250        2141.88   2662.51   57.49   57.77   57.21    0.57
 12     300        1871.97   2242.94   59.23   56.28   62.50    0.59
 14     350        1290.11   1766.11   60.42   58.90   62.02    0.60
 16     400        1084.36   1651.73   67.77   66.82   68.75    0.68
 19     450         930.90   1576.54   61.18   65.75   57.21    0.61
 21     5

[2026-11-07 05:50:15,936] INFO: Training completed. Best model checkpoint: models/ner_trf_trstr_paraphrase/model-best


✔ Saved pipeline to output directory
models/ner_trf_trstr_paraphrase/model-last
TRSTR-Paraphrase model trained. Best checkpoint: models/ner_trf_trstr_paraphrase/model-best


---
## Phase 10: Evaluation (TRSTR-Paraphrase)

Evaluate the TRSTR-Paraphrase model on the same real evaluation partitions.

In [15]:
# Initialize TRSTR-Paraphrase pipeline
pipe_trstr_para = HybridJournalPipeline(
    model_path=trstr_para_best, terms_csv_path="data/terms.csv"
)

# 1. Evaluate on Real Held-Out Test Set
test_trstr_para = evaluate_test_docbin(
    pipe_trstr_para, "data/training/test.spacy", run_label="trstr_paraphrase"
)

# 2. Evaluate on Unseen Benchmark
unseen_trstr_para = evaluate_unseen_generalization(
    pipe_trstr_para, run_label="trstr_paraphrase"
)

# Print TRSTR-Paraphrase Results
print("                    TRSTR-PARAPHRASE RESULTS")
print("=" * 75)
print(f"{'METRIC':<42} | {'TRSTR-Paraphrase':>18}")
print("-" * 75)
print(f"{'Held-Out Real Test Overall F1':<42} | {test_trstr_para['overall_f1']:>16}%")
print(f"{'Held-Out Real Test Overall Precision':<42} | {test_trstr_para['overall_precision']:>16}%")
print(f"{'Held-Out Real Test Overall Recall':<42} | {test_trstr_para['overall_recall']:>16}%")
print("-" * 75)
for lbl in ['IT_TERM', 'CLERICAL_TERM']:
    sl = test_trstr_para['labels'].get(lbl, {})
    print(f"  {lbl + ' F1':<40} | {sl.get('f1', 0):>16}%")
    print(f"  {lbl + ' Precision':<40} | {sl.get('precision', 0):>16}%")
    print(f"  {lbl + ' Recall':<40} | {sl.get('recall', 0):>16}%")
print("-" * 75)

b_trf_p = unseen_trstr_para['modes']['transformer_only']
print(f"{'Unseen Benchmark Transformer Recall':<42} | {b_trf_p.get('recall_pct', 0):>16}%")
print(f"{'Unseen Benchmark Transformer Precision':<42} | {b_trf_p.get('precision_pct', 0):>16}%")
print(f"{'Unseen Benchmark Transformer F1':<42} | {b_trf_p.get('f1_pct', 0):>16}%")
b_hyb_p = unseen_trstr_para['modes']['hybrid']
print(f"{'Unseen Benchmark Hybrid Recall':<42} | {b_hyb_p.get('recall_pct', 0):>16}%")
print(f"{'Unseen Benchmark Hybrid Precision':<42} | {b_hyb_p.get('precision_pct', 0):>16}%")
print(f"{'Unseen Benchmark Hybrid F1':<42} | {b_hyb_p.get('f1_pct', 0):>16}%")
print(f"{'Unseen Benchmark (Generalization Lift)':<42} | {unseen_trstr_para.get('generalization_lift', 'N/A'):>17}")
print("=" * 75)

[2026-11-07 05:50:15,948] INFO: Loaded 372 unique terms from data/terms.csv
[2026-11-07 05:50:15,948] INFO: Loading transformer model from 'models/ner_trf_trstr_paraphrase/model-best'...
[2026-11-07 05:50:19,038] INFO: Configured standalone Dictionary matcher with 372 patterns.
[2026-11-07 05:50:19,593] INFO: [DICTIONARY OVERRIDE] Span 'automated Excel tool' (14:34): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 05:50:22,236] INFO: [DICTIONARY OVERRIDE] Span 'office database' (38:53): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 05:50:23,515] INFO: [DICTIONARY OVERRIDE] Span 'student records' (35:50): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 05:50:25,551] INFO: [DICTIONARY OVERRIDE] Span 'PNP File Management' (7:26): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 05:50:25,873] INFO: [DICTIONARY OVERRIDE] Span 'system records' (23:37): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 05:50:26,001] INFO: [DICTIO

                    TRSTR-PARAPHRASE RESULTS
METRIC                                     |   TRSTR-Paraphrase
---------------------------------------------------------------------------
Held-Out Real Test Overall F1              |            66.22%
Held-Out Real Test Overall Precision       |            69.16%
Held-Out Real Test Overall Recall          |            63.52%
---------------------------------------------------------------------------
  IT_TERM F1                               |            66.03%
  IT_TERM Precision                        |            67.53%
  IT_TERM Recall                           |             64.6%
  CLERICAL_TERM F1                         |            66.67%
  CLERICAL_TERM Precision                  |            73.33%
  CLERICAL_TERM Recall                     |            61.11%
---------------------------------------------------------------------------
Unseen Benchmark Transformer Recall        |             60.0%
Unseen Benchmark Transformer Prec

---
---
# CONDITION 3: TRSTR via LLM Generation (Gemini)
---
## Phase 11: LLM-Direct Synthetic Data Generation

Generate novel sentence + entity pairs using Google Gemini (`gemini-3.5-flash-lite`) via `scripts/generate_llm_synthetic.py`.

Key design:
- LLM generates novel sentences (NOT paraphrases of existing ones)
- Grounded in annotation guidelines, terms.csv, and style examples from real data
- Entity offsets computed post-generation via exact string search (never by LLM)
- Quality filters: generic noun blocklist, benchmark term isolation, dedup guard

In [16]:
from scripts.generate_llm_synthetic import run_generation

LLM_SYNTHETIC_PATH = "data/synthetic_llm_generated.jsonl"
PARAPHRASE_PATH = "data/synthetic_paraphrases.jsonl"

# Dynamically target the exact count of synthetic paraphrases for 1:1 balance
if os.path.exists(PARAPHRASE_PATH):
    with open(PARAPHRASE_PATH, "r", encoding="utf-8") as f:
        TARGET_SYNTHETIC = sum(1 for line in f if line.strip())
else:
    TARGET_SYNTHETIC = 550

BATCH_SIZE = 10
NUM_BATCHES = (TARGET_SYNTHETIC + BATCH_SIZE - 1) // BATCH_SIZE

# Check if synthetic data already exists and is sufficient
existing_count = 0
if os.path.exists(LLM_SYNTHETIC_PATH):
    with open(LLM_SYNTHETIC_PATH, "r", encoding="utf-8") as f:
        existing_count = sum(1 for line in f if line.strip())

if existing_count >= TARGET_SYNTHETIC:
    print(f"Sufficient LLM-generated synthetic data already exists: {existing_count} records (target: {TARGET_SYNTHETIC})")
    print(f"Skipping generation. To regenerate, delete {LLM_SYNTHETIC_PATH} and re-run.")
else:
    print(f"Generating LLM synthetic data ({existing_count} existing, target {TARGET_SYNTHETIC} to match paraphrases)...")
    print(f"This calls the Gemini API and may take 5-10 minutes depending on rate limits.")
    llm_records = run_generation(
        num_batches=NUM_BATCHES,
        batch_size=BATCH_SIZE,
        output_path=LLM_SYNTHETIC_PATH,
        model="gemini-3.5-flash-lite",
    )
    print(f"\nLLM generation complete: {len(llm_records)} records saved.")

Sufficient LLM-generated synthetic data already exists: 571 records (target: 565)
Skipping generation. To regenerate, delete data/synthetic_llm_generated.jsonl and re-run.


---
## Phase 12: Build TRSTR-LLM Training Dataset & Train

Combine real training records with LLM-generated synthetic records (and optionally mined hard negatives), then fine-tune a fresh transformer.

In [17]:
from scripts.prepare_trstr_llm import build_trstr_llm_dataset

trstr_llm_result = build_trstr_llm_dataset(
    train_spacy_path="data/training/train.spacy",
    synthetic_jsonl_path=LLM_SYNTHETIC_PATH,
    mined_negatives_path="data/review/mined_hard_negatives.jsonl",
    dev_spacy_path="data/training/dev.spacy",
    test_spacy_path="data/training/test.spacy",
    terms_csv_path="data/terms.csv",
    output_jsonl_path="data/training_trstr_llm.jsonl",
    output_spacy_path="data/training/train_trstr_llm.spacy",
)

print(f"\nTRSTR-LLM Training Pool Summary:")
print(f"  Total records      : {trstr_llm_result['total_records']}")
print(f"  Real records       : {trstr_llm_result['real_count']}")
print(f"  Synthetic records  : {trstr_llm_result['synthetic_count']}")
print(f"  Mined negatives    : {trstr_llm_result['mined_count']}")
print(f"  Negative ratio     : {trstr_llm_result['negative_ratio_pct']}%")
print(f"  Output DocBin      : {trstr_llm_result['output_spacy']}")

[2026-11-07 05:50:34,984] INFO: Loaded 923 real training documents from data/training/train.spacy
[2026-11-07 05:50:34,994] INFO: Loaded 571 synthetic records from data/synthetic_llm_generated.jsonl
[2026-11-07 05:50:34,996] INFO: Loaded 164 raw mined hard negative entries from data/review/mined_hard_negatives.jsonl
[2026-11-07 05:50:35,064] INFO: Mined negatives curation audit: 164 raw -> 24 intra-file duplicates removed, 19 held-out test leaks blocked, 15 dev split leaks blocked, 12 accidental entity sentences filtered -> 94 clean hard negatives accepted.
[2026-11-07 05:50:35,065] INFO: Combined TRSTR-LLM pool: 1588 total records (923 real + 571 synthetic + 94 mined negatives)
[2026-11-07 05:50:35,075] INFO: Saved 1588 records to data/training_trstr_llm.jsonl
[2026-11-07 05:50:35,218] INFO: Saved combined DocBin to data/training/train_trstr_llm.spacy (1588 total docs, 0 synthetic entities dropped)
[2026-11-07 05:50:35,219] INFO: --- Real Training Split Diagnostics ---
[2026-11-07 05:


TRSTR-LLM Training Pool Summary:
  Total records      : 1588
  Real records       : 923
  Synthetic records  : 571
  Mined negatives    : 94
  Negative ratio     : 37.03%
  Output DocBin      : data/training/train_trstr_llm.spacy


In [18]:
# Free GPU memory and reload training module
import gc
import torch
import importlib
import scripts.training
importlib.reload(scripts.training)
from scripts.training import train_ner_trf

for var_name in ["pipe_trtr", "pipe_trstr_para", "pipe_trstr_llm", "para_model", "para_tokenizer"]:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Train TRSTR-LLM model from scratch
trstr_llm_model_dir = "models/ner_trf_trstr_llm"
trstr_llm_best = os.path.join(trstr_llm_model_dir, "model-best")

# Clean previous model to train from scratch
if os.path.exists(trstr_llm_model_dir):
    shutil.rmtree(trstr_llm_model_dir)
    print(f"Cleaned previous TRSTR-LLM model at {trstr_llm_model_dir}")

print("Fine-tuning TRSTR-LLM model...")
train_ner_trf(
    output_dir=trstr_llm_model_dir,
    train_path="data/training/train_trstr_llm.spacy",
    dev_path="data/training/dev.spacy",
    config_path="config_trf.cfg",
    use_gpu=0,
    max_steps=2500,
    eval_frequency=50,
    patience=400
)
print(f"TRSTR-LLM model trained. Best checkpoint: {trstr_llm_best}")


[2026-11-07 05:50:36,411] INFO: Starting transformer training on GPU 0 for up to 2500 steps (eval_frequency=50, patience=400 steps, max_batch_items=2048)...
[2026-11-07 05:50:36,507] [INFO] Set up nlp object from config
[2026-11-07 05:50:36,507] INFO: Set up nlp object from config
[2026-11-07 05:50:36,518] [INFO] Pipeline: ['transformer', 'ner']
[2026-11-07 05:50:36,518] INFO: Pipeline: ['transformer', 'ner']
[2026-11-07 05:50:36,523] [INFO] Created vocabulary
[2026-11-07 05:50:36,523] INFO: Created vocabulary
[2026-11-07 05:50:36,523] [INFO] Finished initializing nlp object
[2026-11-07 05:50:36,523] INFO: Finished initializing nlp object


Cleaned previous TRSTR-LLM model at models/ner_trf_trstr_llm
Fine-tuning TRSTR-LLM model...
ℹ Saving to output directory: models/ner_trf_trstr_llm
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[2026-11-07 05:50:38,363] [INFO] Initialized pipeline components: ['transformer', 'ner']
[2026-11-07 05:50:38,363] INFO: Initialized pipeline components: ['transformer', 'ner']


✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  -------------  --------  ------  ------  ------  ------
  0       0        3389.47    189.55    0.00    0.00    0.00    0.00
  1      50       97725.92  27693.87    0.00    0.00    0.00    0.00
  3     100        3700.50   5056.39   29.63   36.36   25.00    0.30
  5     150        3386.53   4017.66   40.00   42.86   37.50    0.40
  7     200        2734.68   3372.80   46.11   52.12   41.35    0.46
  9     250        1971.57   2590.11   54.36   56.48   52.40    0.54
 11     300        1673.94   2330.78   53.81   53.30   54.33    0.54
 13     350        1285.95   1871.15   56.94   54.91   59.13    0.57
 15     400        1224.62   1820.28   57.86   60.10   55.77    0.58
 17     450         901.64   1448.67   61.11   58.93   63.46    0.61
 19     5

[2026-11-07 05:58:03,920] INFO: Training completed. Best model checkpoint: models/ner_trf_trstr_llm/model-best


✔ Saved pipeline to output directory
models/ner_trf_trstr_llm/model-last
TRSTR-LLM model trained. Best checkpoint: models/ner_trf_trstr_llm/model-best


---
## Phase 12b: Distil-Model (TRSTR-LLM Mini) — `trstr_llm_mini`

Train a **lightweight, drop-in version** of the TRSTR-LLM model using `distilroberta-base` as the backbone in place of `roberta-base`. `distilroberta-base` is the knowledge-distilled sibling of `roberta-base`: it shares the **identical BPE vocabulary and tokenizer**, is **case-sensitive**, and preserves the same architectural depth in a smaller form (6 layers, 82M params vs 125M).

| | `roberta-base` (original) | `distilroberta-base` (mini) |
|---|---|---|
| Parameters | 125M | 82M (~34% fewer) |
| Disk size (FP32) | ~481 MB | ~330 MB |
| Inference RAM (FP16) | ~350–500 MB | **~230–280 MB** |
| Tokenizer | BPE, case-sensitive | BPE, case-sensitive ✓ |

The model is trained on the **same `train_trstr_llm.spacy` dataset** built in Phase 12, using an identical hyperparameter schedule via `config_trf_mini.cfg`. No capability is lost — both models recognise `IT_TERM` and `CLERICAL_TERM` entities via the same hybrid pipeline.

In [19]:
# Free GPU memory before mini-model training
import gc
import torch
import importlib
import scripts.training
importlib.reload(scripts.training)
from scripts.training import train_ner_trf

for var_name in ["pipe_trtr", "pipe_trstr_para", "pipe_trstr_llm", "pipe_trstr_llm_mini", "para_model", "para_tokenizer"]:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# --- Train trstr_llm_mini (distilroberta-base backbone) ---
# Re-uses the TRSTR-LLM dataset already built in Phase 12 (no new data generation needed).
TRSTR_LLM_MINI_DIR = "models/trstr_llm_mini"
trstr_llm_mini_best = os.path.join(TRSTR_LLM_MINI_DIR, "model-best")

# Clean previous mini-model to train from scratch
if os.path.exists(TRSTR_LLM_MINI_DIR):
    shutil.rmtree(TRSTR_LLM_MINI_DIR)
    print(f"Cleaned previous trstr_llm_mini model at {TRSTR_LLM_MINI_DIR}")

print("Fine-tuning trstr_llm_mini (distilroberta-base backbone)...")
train_ner_trf(
    output_dir=TRSTR_LLM_MINI_DIR,
    train_path="data/training/train_trstr_llm.spacy",   # same dataset as full model
    dev_path="data/training/dev.spacy",
    config_path="config_trf_mini.cfg",                  # distilroberta-base config
    use_gpu=0,
    max_steps=2500,
    eval_frequency=50,
    patience=400,
)
print(f"trstr_llm_mini trained. Best checkpoint: {trstr_llm_mini_best}")


[2026-11-07 05:58:05,397] INFO: Starting transformer training on GPU 0 for up to 2500 steps (eval_frequency=50, patience=400 steps, max_batch_items=2048)...
[2026-11-07 05:58:05,508] [INFO] Set up nlp object from config
[2026-11-07 05:58:05,508] INFO: Set up nlp object from config
[2026-11-07 05:58:05,521] [INFO] Pipeline: ['transformer', 'ner']
[2026-11-07 05:58:05,521] INFO: Pipeline: ['transformer', 'ner']
[2026-11-07 05:58:05,526] [INFO] Created vocabulary
[2026-11-07 05:58:05,526] INFO: Created vocabulary
[2026-11-07 05:58:05,527] [INFO] Finished initializing nlp object
[2026-11-07 05:58:05,527] INFO: Finished initializing nlp object


Cleaned previous trstr_llm_mini model at models/trstr_llm_mini
Fine-tuning trstr_llm_mini (distilroberta-base backbone)...
ℹ Saving to output directory: models/trstr_llm_mini
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================


[2026-11-07 05:58:07,078] [INFO] Initialized pipeline components: ['transformer', 'ner']
[2026-11-07 05:58:07,078] INFO: Initialized pipeline components: ['transformer', 'ner']


✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  -------------  --------  ------  ------  ------  ------
  0       0        1875.52    171.18    0.35    0.28    0.48    0.00
  1      50      120405.69  31317.00    0.00    0.00    0.00    0.00
  3     100        3893.97   5511.39   15.38   38.46    9.62    0.15
  5     150        3792.36   4529.99   35.83   33.91   37.98    0.36
  7     200        2678.06   3397.05   41.44   48.70   36.06    0.41
  9     250        2153.97   2763.73   50.25   53.23   47.60    0.50
 11     300        1835.14   2562.92   48.73   51.61   46.15    0.49
 13     350        1491.06   2080.38   57.58   52.36   63.94    0.58
 15     400        1352.01   2035.05   55.50   57.81   53.37    0.55
 17     450        1003.43   1641.82   58.22   56.88   59.62    0.58
 19     5

[2026-11-07 06:08:18,368] INFO: Training completed. Best model checkpoint: models/trstr_llm_mini/model-best


✔ Saved pipeline to output directory
models/trstr_llm_mini/model-last
trstr_llm_mini trained. Best checkpoint: models/trstr_llm_mini/model-best


In [20]:
# Initialize trstr_llm_mini pipeline
pipe_trstr_llm_mini = HybridJournalPipeline(
    model_path=trstr_llm_mini_best, terms_csv_path="data/terms.csv"
)

# 1. Evaluate on Real Held-Out Test Set
test_trstr_llm_mini = evaluate_test_docbin(
    pipe_trstr_llm_mini, "data/training/test.spacy", run_label="trstr_llm_mini"
)

# 2. Evaluate on Unseen Benchmark
unseen_trstr_llm_mini = evaluate_unseen_generalization(
    pipe_trstr_llm_mini, run_label="trstr_llm_mini"
)

# --- Memory footprint & RSS measurement ---
try:
    import os as _os
    _model_size_bytes = sum(
        _os.path.getsize(_os.path.join(dp, f))
        for dp, _, fns in _os.walk(trstr_llm_mini_best)
        for f in fns
    )
    _model_size_mb = _model_size_bytes / (1024 ** 2)
except Exception:
    _model_size_mb = float("nan")

try:
    import os as _os
    import psutil as _psutil
    _rss_mb = _psutil.Process(_os.getpid()).memory_info().rss / 1_048_576
except Exception:
    _rss_mb = float("nan")

# Print trstr_llm_mini Results
print("                TRSTR-LLM-MINI RESULTS  (distilroberta-base)")
print("=" * 75)
print(f"{'METRIC':<42} | {'TRSTR-LLM-MINI':>18}")
print("-" * 75)
print(f"{'Model size on disk':<42} | {f'{_model_size_mb:.1f} MB':>18}")
print(f"{'Process RSS (Memory after eval)':<42} | {f'{_rss_mb:.1f} MB':>18}")
print("-" * 75)
print(f"{'Held-Out Real Test Overall F1':<42} | {test_trstr_llm_mini['overall_f1']:>16}%")
print(f"{'Held-Out Real Test Overall Precision':<42} | {test_trstr_llm_mini['overall_precision']:>16}%")
print(f"{'Held-Out Real Test Overall Recall':<42} | {test_trstr_llm_mini['overall_recall']:>16}%")
print("-" * 75)
for lbl in ['IT_TERM', 'CLERICAL_TERM']:
    sl = test_trstr_llm_mini['labels'].get(lbl, {})
    print(f"  {lbl + ' F1':<40} | {sl.get('f1', 0):>16}%")
    print(f"  {lbl + ' Precision':<40} | {sl.get('precision', 0):>16}%")
    print(f"  {lbl + ' Recall':<40} | {sl.get('recall', 0):>16}%")
print("-" * 75)

b_trf_m = unseen_trstr_llm_mini['modes']['transformer_only']
print(f"{'Unseen Benchmark Transformer Recall':<42} | {b_trf_m.get('recall_pct', 0):>16}%")
print(f"{'Unseen Benchmark Transformer Precision':<42} | {b_trf_m.get('precision_pct', 0):>16}%")
print(f"{'Unseen Benchmark Transformer F1':<42} | {b_trf_m.get('f1_pct', 0):>16}%")
b_hyb_m = unseen_trstr_llm_mini['modes']['hybrid']
print(f"{'Unseen Benchmark Hybrid Recall':<42} | {b_hyb_m.get('recall_pct', 0):>16}%")
print(f"{'Unseen Benchmark Hybrid Precision':<42} | {b_hyb_m.get('precision_pct', 0):>16}%")
print(f"{'Unseen Benchmark Hybrid F1':<42} | {b_hyb_m.get('f1_pct', 0):>16}%")
print(f"{'Unseen Benchmark (Generalization Lift)':<42} | {unseen_trstr_llm_mini.get('generalization_lift', 'N/A'):>17}")
print("=" * 75)


[2026-11-07 06:08:18,379] INFO: Loaded 372 unique terms from data/terms.csv
[2026-11-07 06:08:18,380] INFO: Loading transformer model from 'models/trstr_llm_mini/model-best'...
[2026-11-07 06:08:20,412] INFO: Configured standalone Dictionary matcher with 372 patterns.
[2026-11-07 06:08:20,850] INFO: [DICTIONARY OVERRIDE] Span 'Excel tool' (24:34): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 06:08:22,804] INFO: [DICTIONARY OVERRIDE] Span 'office database' (38:53): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 06:08:25,439] INFO: [DICTIONARY OVERRIDE] Span 'database' (43:51): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 06:08:26,236] INFO: [DICTIONARY OVERRIDE] Span 'vendor evaluation' (10:27): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 06:08:26,275] INFO: [DICTIONARY OVERRIDE] Span 'Record management systems' (0:25): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 06:08:27,231] INFO: Per-item results: data/ev

                TRSTR-LLM-MINI RESULTS  (distilroberta-base)
METRIC                                     |     TRSTR-LLM-MINI
---------------------------------------------------------------------------
Model size on disk                         |           318.2 MB
Process RSS (Memory after eval)            |          1635.5 MB
---------------------------------------------------------------------------
Held-Out Real Test Overall F1              |            70.33%
Held-Out Real Test Overall Precision       |            72.07%
Held-Out Real Test Overall Recall          |            68.67%
---------------------------------------------------------------------------
  IT_TERM F1                               |            70.29%
  IT_TERM Precision                        |            72.37%
  IT_TERM Recall                           |            68.32%
  CLERICAL_TERM F1                         |            70.42%
  CLERICAL_TERM Precision                  |            71.43%
  CLERICAL_TERM

---
## Phase 12c: Ultra-Lightweight CNN Alternative (TRSTR-LLM CNN) — `trstr_llm_cnn`

For hardware environments where RAM is strictly constrained to **≤ 512 MB** or even **≤ 256 MB**, a non-transformer **`tok2vec` (CNN) architecture** provides an extreme memory reduction:

| Architecture | Backbone | Disk Size | Production Inference RAM |
|---|---|---|---|
| **TRSTR-LLM (Original)** | `roberta-base` | ~481 MB | ~450–550 MB |
| **TRSTR-LLM Mini** | `distilroberta-base` | ~330 MB | **~250–320 MB** |
| **TRSTR-LLM CNN** | `spacy.Tok2Vec.v2` (MultiHashEmbed + MaxoutWindowEncoder) | **~15 MB** | **~25–40 MB** |

The CNN model uses **zero static word vectors** (relies entirely on sub-token hash embeddings across `NORM`, `PREFIX`, `SUFFIX`, and `SHAPE`) and requires **no PyTorch runtime at inference time**, leaving over 400 MB of free memory headroom on a 512 MB VPS.

In [21]:
# Free GPU memory before CNN model training
import gc
import torch
import importlib
import scripts.training
importlib.reload(scripts.training)
from scripts.training import train_ner_cnn

for var_name in ["pipe_trtr", "pipe_trstr_para", "pipe_trstr_llm", "pipe_trstr_llm_mini", "pipe_trstr_llm_cnn"]:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# --- Train trstr_llm_cnn (Tok2Vec CNN backbone) ---
TRSTR_LLM_CNN_DIR = "models/trstr_llm_cnn"
trstr_llm_cnn_best = os.path.join(TRSTR_LLM_CNN_DIR, "model-best")

if os.path.exists(TRSTR_LLM_CNN_DIR):
    shutil.rmtree(TRSTR_LLM_CNN_DIR)
    print(f"Cleaned previous trstr_llm_cnn model at {TRSTR_LLM_CNN_DIR}")

print("Training trstr_llm_cnn (pure spaCy tok2vec CNN, <40MB RAM target)...")
train_ner_cnn(
    output_dir=TRSTR_LLM_CNN_DIR,
    train_path="data/training/train_trstr_llm.spacy",
    dev_path="data/training/dev.spacy",
    config_path="config_cnn_mini.cfg",
    use_gpu=0,
    max_steps=5000,
    eval_frequency=100,
    patience=800,
)
print(f"trstr_llm_cnn trained. Checkpoint: {trstr_llm_cnn_best}")


[2026-11-07 06:08:32,200] INFO: Starting CNN tok2vec training for up to 5000 steps...
[2026-11-07 06:08:32,328] [INFO] Set up nlp object from config
[2026-11-07 06:08:32,328] INFO: Set up nlp object from config
[2026-11-07 06:08:32,343] [INFO] Pipeline: ['tok2vec', 'ner']
[2026-11-07 06:08:32,343] INFO: Pipeline: ['tok2vec', 'ner']
[2026-11-07 06:08:32,347] [INFO] Created vocabulary
[2026-11-07 06:08:32,347] INFO: Created vocabulary
[2026-11-07 06:08:32,348] [INFO] Finished initializing nlp object
[2026-11-07 06:08:32,348] INFO: Finished initializing nlp object


Cleaned previous trstr_llm_cnn model at models/trstr_llm_cnn
Training trstr_llm_cnn (pure spaCy tok2vec CNN, <40MB RAM target)...
ℹ Saving to output directory: models/trstr_llm_cnn
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================


[2026-11-07 06:08:33,095] [INFO] Initialized pipeline components: ['tok2vec', 'ner']
[2026-11-07 06:08:33,095] INFO: Initialized pipeline components: ['tok2vec', 'ner']


✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     38.00    0.00    0.00    0.00    0.00
  0     100         51.05   1416.81    7.38   12.22    5.29    0.07
  0     200        489.53   1024.76   23.90   34.55   18.27    0.24
  1     300         46.29    781.38   35.36   39.18   32.21    0.35
  1     400         84.63    698.49   47.87   53.57   43.27    0.48
  2     500         84.63    735.43   57.00   59.38   54.81    0.57
  2     600        101.97    591.10   55.07   55.34   54.81    0.55
  3     700        243.59    540.21   54.59   60.12   50.00    0.55
  4     800        138.97    557.95   57.07   61.33   53.37    0.57
  5     900        157.17    453.28   59.30   62.11   56.73    0.59
  5    1000        153.

[2026-11-07 06:13:01,263] INFO: CNN training completed. Checkpoint: models/trstr_llm_cnn/model-best


✔ Saved pipeline to output directory
models/trstr_llm_cnn/model-last
trstr_llm_cnn trained. Checkpoint: models/trstr_llm_cnn/model-best


In [22]:
# Initialize trstr_llm_cnn pipeline
pipe_trstr_llm_cnn = HybridJournalPipeline(
    model_path=trstr_llm_cnn_best, terms_csv_path="data/terms.csv"
)

# 1. Evaluate on Real Held-Out Test Set
test_trstr_llm_cnn = evaluate_test_docbin(
    pipe_trstr_llm_cnn, "data/training/test.spacy", run_label="trstr_llm_cnn"
)

# 2. Evaluate on Unseen Benchmark
unseen_trstr_llm_cnn = evaluate_unseen_generalization(
    pipe_trstr_llm_cnn, run_label="trstr_llm_cnn"
)

# --- Memory footprint & RSS measurement ---
try:
    import os as _os
    _model_size_bytes = sum(
        _os.path.getsize(_os.path.join(dp, f))
        for dp, _, fns in _os.walk(trstr_llm_cnn_best)
        for f in fns
    )
    _model_size_mb = _model_size_bytes / (1024 ** 2)
except Exception:
    _model_size_mb = float("nan")

try:
    import os as _os
    import psutil as _psutil
    _rss_mb = _psutil.Process(_os.getpid()).memory_info().rss / 1_048_576
except Exception:
    _rss_mb = float("nan")

# Print trstr_llm_cnn Results
print("                TRSTR-LLM-CNN RESULTS  (tok2vec CNN)")
print("=" * 75)
print(f"{'METRIC':<42} | {'TRSTR-LLM-CNN':>18}")
print("-" * 75)
print(f"{'Model size on disk':<42} | {f'{_model_size_mb:.1f} MB':>18}")
print(f"{'Process RSS (Memory after eval)':<42} | {f'{_rss_mb:.1f} MB':>18}")
print("-" * 75)
print(f"{'Held-Out Real Test Overall F1':<42} | {test_trstr_llm_cnn['overall_f1']:>16}%")
print(f"{'Held-Out Real Test Overall Precision':<42} | {test_trstr_llm_cnn['overall_precision']:>16}%")
print(f"{'Held-Out Real Test Overall Recall':<42} | {test_trstr_llm_cnn['overall_recall']:>16}%")
print("-" * 75)
for lbl in ['IT_TERM', 'CLERICAL_TERM']:
    sl = test_trstr_llm_cnn['labels'].get(lbl, {})
    print(f"  {lbl + ' F1':<40} | {sl.get('f1', 0):>16}%")
    print(f"  {lbl + ' Precision':<40} | {sl.get('precision', 0):>16}%")
    print(f"  {lbl + ' Recall':<40} | {sl.get('recall', 0):>16}%")
print("-" * 75)

b_trf_m = unseen_trstr_llm_cnn['modes']['transformer_only']
print(f"{'Unseen Benchmark Model Recall':<42} | {b_trf_m.get('recall_pct', 0):>16}%")
print(f"{'Unseen Benchmark Model Precision':<42} | {b_trf_m.get('precision_pct', 0):>16}%")
print(f"{'Unseen Benchmark Model F1':<42} | {b_trf_m.get('f1_pct', 0):>16}%")
b_hyb_m = unseen_trstr_llm_cnn['modes']['hybrid']
print(f"{'Unseen Benchmark Hybrid Recall':<42} | {b_hyb_m.get('recall_pct', 0):>16}%")
print(f"{'Unseen Benchmark Hybrid Precision':<42} | {b_hyb_m.get('precision_pct', 0):>16}%")
print(f"{'Unseen Benchmark Hybrid F1':<42} | {b_hyb_m.get('f1_pct', 0):>16}%")
print(f"{'Unseen Benchmark (Generalization Lift)':<42} | {unseen_trstr_llm_cnn.get('generalization_lift', 'N/A'):>17}")
print("=" * 75)


[2026-11-07 06:13:01,275] INFO: Loaded 372 unique terms from data/terms.csv
[2026-11-07 06:13:01,276] INFO: Loading transformer model from 'models/trstr_llm_cnn/model-best'...
[2026-11-07 06:13:01,537] INFO: Configured standalone Dictionary matcher with 372 patterns.
[2026-11-07 06:13:02,530] INFO: [DICTIONARY OVERRIDE] Span 'prefect clearance portal' (13:37): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 06:13:03,006] INFO: [DICTIONARY OVERRIDE] Span 'system records' (23:37): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 06:13:03,037] INFO: [DICTIONARY OVERRIDE] Span 'database' (43:51): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 06:13:03,201] INFO: [DICTIONARY OVERRIDE] Span 'database' (15:23): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 06:13:03,248] INFO: [DICTIONARY OVERRIDE] Span 'Record management systems' (0:25): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 06:13:03,294] INFO: [DICTIONARY OVERRIDE] 

                TRSTR-LLM-CNN RESULTS  (tok2vec CNN)
METRIC                                     |      TRSTR-LLM-CNN
---------------------------------------------------------------------------
Model size on disk                         |             6.0 MB
Process RSS (Memory after eval)            |          1638.1 MB
---------------------------------------------------------------------------
Held-Out Real Test Overall F1              |            57.96%
Held-Out Real Test Overall Precision       |            64.89%
Held-Out Real Test Overall Recall          |            52.36%
---------------------------------------------------------------------------
  IT_TERM F1                               |            59.31%
  IT_TERM Precision                        |            66.67%
  IT_TERM Recall                           |            53.42%
  CLERICAL_TERM F1                         |            54.96%
  CLERICAL_TERM Precision                  |            61.02%
  CLERICAL_TERM Recall 

---
## Phase 13: Evaluation (TRSTR-LLM)

Evaluate the TRSTR-LLM model on the same real evaluation partitions.

In [23]:
# Initialize TRSTR-LLM pipeline
pipe_trstr_llm = HybridJournalPipeline(
    model_path=trstr_llm_best, terms_csv_path="data/terms.csv"
)

# 1. Evaluate on Real Held-Out Test Set
test_trstr_llm = evaluate_test_docbin(
    pipe_trstr_llm, "data/training/test.spacy", run_label="trstr_llm"
)

# 2. Evaluate on Unseen Benchmark
unseen_trstr_llm = evaluate_unseen_generalization(
    pipe_trstr_llm, run_label="trstr_llm"
)

# Print TRSTR-LLM Results
print("                    TRSTR-LLM RESULTS")
print("=" * 75)
print(f"{'METRIC':<42} | {'TRSTR-LLM':>18}")
print("-" * 75)
print(f"{'Held-Out Real Test Overall F1':<42} | {test_trstr_llm['overall_f1']:>16}%")
print(f"{'Held-Out Real Test Overall Precision':<42} | {test_trstr_llm['overall_precision']:>16}%")
print(f"{'Held-Out Real Test Overall Recall':<42} | {test_trstr_llm['overall_recall']:>16}%")
print("-" * 75)
for lbl in ['IT_TERM', 'CLERICAL_TERM']:
    sl = test_trstr_llm['labels'].get(lbl, {})
    print(f"  {lbl + ' F1':<40} | {sl.get('f1', 0):>16}%")
    print(f"  {lbl + ' Precision':<40} | {sl.get('precision', 0):>16}%")
    print(f"  {lbl + ' Recall':<40} | {sl.get('recall', 0):>16}%")
print("-" * 75)

b_trf_l = unseen_trstr_llm['modes']['transformer_only']
print(f"{'Unseen Benchmark Transformer Recall':<42} | {b_trf_l.get('recall_pct', 0):>16}%")
print(f"{'Unseen Benchmark Transformer Precision':<42} | {b_trf_l.get('precision_pct', 0):>16}%")
print(f"{'Unseen Benchmark Transformer F1':<42} | {b_trf_l.get('f1_pct', 0):>16}%")
b_hyb_l = unseen_trstr_llm['modes']['hybrid']
print(f"{'Unseen Benchmark Hybrid Recall':<42} | {b_hyb_l.get('recall_pct', 0):>16}%")
print(f"{'Unseen Benchmark Hybrid Precision':<42} | {b_hyb_l.get('precision_pct', 0):>16}%")
print(f"{'Unseen Benchmark Hybrid F1':<42} | {b_hyb_l.get('f1_pct', 0):>16}%")
print(f"{'Unseen Benchmark (Generalization Lift)':<42} | {unseen_trstr_llm.get('generalization_lift', 'N/A'):>17}")
print("=" * 75)

[2026-11-07 06:13:04,385] INFO: Loaded 372 unique terms from data/terms.csv
[2026-11-07 06:13:04,386] INFO: Loading transformer model from 'models/ner_trf_trstr_llm/model-best'...
[2026-11-07 06:13:06,974] INFO: Configured standalone Dictionary matcher with 372 patterns.
[2026-11-07 06:13:07,446] INFO: [DICTIONARY OVERRIDE] Span 'automated Excel tool' (14:34): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 06:13:09,626] INFO: [DICTIONARY OVERRIDE] Span 'office database' (38:53): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 06:13:12,816] INFO: [DICTIONARY OVERRIDE] Span 'database' (43:51): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 06:13:13,478] INFO: [DICTIONARY OVERRIDE] Span 'data validation' (12:27): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 06:13:13,806] INFO: [DICTIONARY OVERRIDE] Span 'Record management systems' (0:25): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 06:13:14,649] INFO: [DICTIONARY OV

                    TRSTR-LLM RESULTS
METRIC                                     |          TRSTR-LLM
---------------------------------------------------------------------------
Held-Out Real Test Overall F1              |            64.85%
Held-Out Real Test Overall Precision       |            68.75%
Held-Out Real Test Overall Recall          |            61.37%
---------------------------------------------------------------------------
  IT_TERM F1                               |            64.65%
  IT_TERM Precision                        |            70.59%
  IT_TERM Recall                           |            59.63%
  CLERICAL_TERM F1                         |            65.28%
  CLERICAL_TERM Precision                  |            65.28%
  CLERICAL_TERM Recall                     |            65.28%
---------------------------------------------------------------------------
Unseen Benchmark Transformer Recall        |            86.15%
Unseen Benchmark Transformer Precision  

---
---
# COMPARATIVE ANALYSIS
---
## Phase 14: Side-by-Side Comparative Evaluation Table

Compare all 3 conditions (TRTR, TRSTR-Paraphrase, TRSTR-LLM) on the same held-out test set and unseen benchmark.

In [24]:
def fmt(v, suffix="%"):
    """Format a numeric value for table display."""
    if isinstance(v, (int, float)):
        return f"{v}{suffix}"
    return str(v)

print("\n" + "=" * 100)
print("          SIDE-BY-SIDE COMPARATIVE EVALUATION — ALL 3 CONDITIONS")
print("=" * 100)
print(f"{'EVALUATION METRIC':<45} | {'TRTR':>14} | {'TRSTR-Para':>14} | {'TRSTR-LLM':>14}")
print("-" * 100)

# Held-out real test set
print(f"{'Held-Out Test Overall F1':<45} | {fmt(test_trtr['overall_f1']):>14} | {fmt(test_trstr_para['overall_f1']):>14} | {fmt(test_trstr_llm['overall_f1']):>14}")
print(f"{'Held-Out Test Overall Precision':<45} | {fmt(test_trtr['overall_precision']):>14} | {fmt(test_trstr_para['overall_precision']):>14} | {fmt(test_trstr_llm['overall_precision']):>14}")
print(f"{'Held-Out Test Overall Recall':<45} | {fmt(test_trtr['overall_recall']):>14} | {fmt(test_trstr_para['overall_recall']):>14} | {fmt(test_trstr_llm['overall_recall']):>14}")
print("-" * 100)

# Per-label
for lbl in ['IT_TERM', 'CLERICAL_TERM']:
    for metric_key, metric_name in [('f1', 'F1'), ('precision', 'Precision'), ('recall', 'Recall')]:
        v_trtr = test_trtr['labels'].get(lbl, {}).get(metric_key, 0)
        v_para = test_trstr_para['labels'].get(lbl, {}).get(metric_key, 0)
        v_llm = test_trstr_llm['labels'].get(lbl, {}).get(metric_key, 0)
        print(f"  {lbl + ' ' + metric_name:<43} | {fmt(v_trtr):>14} | {fmt(v_para):>14} | {fmt(v_llm):>14}")
print("-" * 100)

# Unseen benchmark - Transformer only
trf_trtr = unseen_trtr['modes']['transformer_only']
trf_para = unseen_trstr_para['modes']['transformer_only']
trf_llm = unseen_trstr_llm['modes']['transformer_only']

print(f"{'Unseen Benchmark TRF Recall':<45} | {fmt(trf_trtr.get('recall_pct', 0)):>14} | {fmt(trf_para.get('recall_pct', 0)):>14} | {fmt(trf_llm.get('recall_pct', 0)):>14}")
print(f"{'Unseen Benchmark TRF Precision':<45} | {fmt(trf_trtr.get('precision_pct', 0)):>14} | {fmt(trf_para.get('precision_pct', 0)):>14} | {fmt(trf_llm.get('precision_pct', 0)):>14}")
print(f"{'Unseen Benchmark TRF F1':<45} | {fmt(trf_trtr.get('f1_pct', 0)):>14} | {fmt(trf_para.get('f1_pct', 0)):>14} | {fmt(trf_llm.get('f1_pct', 0)):>14}")
print("-" * 100)

# Unseen benchmark - Hybrid
hyb_trtr = unseen_trtr['modes']['hybrid']
hyb_para = unseen_trstr_para['modes']['hybrid']
hyb_llm = unseen_trstr_llm['modes']['hybrid']

print(f"{'Unseen Benchmark Hybrid Recall':<45} | {fmt(hyb_trtr.get('recall_pct', 0)):>14} | {fmt(hyb_para.get('recall_pct', 0)):>14} | {fmt(hyb_llm.get('recall_pct', 0)):>14}")
print(f"{'Unseen Benchmark Hybrid Precision':<45} | {fmt(hyb_trtr.get('precision_pct', 0)):>14} | {fmt(hyb_para.get('precision_pct', 0)):>14} | {fmt(hyb_llm.get('precision_pct', 0)):>14}")
print(f"{'Unseen Benchmark Hybrid F1':<45} | {fmt(hyb_trtr.get('f1_pct', 0)):>14} | {fmt(hyb_para.get('f1_pct', 0)):>14} | {fmt(hyb_llm.get('f1_pct', 0)):>14}")
print(f"{'Generalization Lift':<45} | {unseen_trtr.get('generalization_lift', 'N/A'):>14} | {unseen_trstr_para.get('generalization_lift', 'N/A'):>14} | {unseen_trstr_llm.get('generalization_lift', 'N/A'):>14}")

print("=" * 100)

# Compute deltas vs TRTR baseline
print("\n" + "-" * 100)
print("  DELTA vs. TRTR BASELINE")
print("-" * 100)
for name, test_res, unseen_res in [
    ("TRSTR-Paraphrase", test_trstr_para, unseen_trstr_para),
    ("TRSTR-LLM", test_trstr_llm, unseen_trstr_llm),
]:
    delta_f1 = round(test_res['overall_f1'] - test_trtr['overall_f1'], 2)
    delta_p = round(test_res['overall_precision'] - test_trtr['overall_precision'], 2)
    delta_r = round(test_res['overall_recall'] - test_trtr['overall_recall'], 2)
    
    trf_mode = unseen_res['modes']['transformer_only']
    trf_base = unseen_trtr['modes']['transformer_only']
    delta_unseen_r = round(trf_mode.get('recall_pct', 0) - trf_base.get('recall_pct', 0), 2)
    delta_unseen_f1 = round(trf_mode.get('f1_pct', 0) - trf_base.get('f1_pct', 0), 2)
    
    sign = lambda x: f"+{x}" if x > 0 else str(x)
    print(f"  {name:<20}: Test F1 {sign(delta_f1)}% | Test P {sign(delta_p)}% | "
          f"Test R {sign(delta_r)}% | Unseen TRF Recall {sign(delta_unseen_r)}% | "
          f"Unseen TRF F1 {sign(delta_unseen_f1)}%")

print("=" * 100)


          SIDE-BY-SIDE COMPARATIVE EVALUATION — ALL 3 CONDITIONS
EVALUATION METRIC                             |           TRTR |     TRSTR-Para |      TRSTR-LLM
----------------------------------------------------------------------------------------------------
Held-Out Test Overall F1                      |         71.27% |         66.22% |         64.85%
Held-Out Test Overall Precision               |         71.74% |         69.16% |         68.75%
Held-Out Test Overall Recall                  |         70.82% |         63.52% |         61.37%
----------------------------------------------------------------------------------------------------
  IT_TERM F1                                  |         71.52% |         66.03% |         64.65%
  IT_TERM Precision                           |          72.9% |         67.53% |         70.59%
  IT_TERM Recall                              |         70.19% |          64.6% |         59.63%
  CLERICAL_TERM F1                            |      

---
## Phase 15: Save Full Evaluation Reports

Export comprehensive JSON reports for each condition.

In [25]:
# Save comprehensive comparison report
comparison_report = {
    "conditions": {
        "TRTR": {
            "description": "Train Real, Test Real — baseline with authentic data only",
            "training_records": len(train_records),
            "held_out_test": test_trtr,
            "unseen_benchmark": {
                "transformer_only": unseen_trtr['modes']['transformer_only'],
                "hybrid": unseen_trtr['modes']['hybrid'],
                "generalization_lift": unseen_trtr.get('generalization_lift'),
            },
        },
        "TRSTR_Paraphrase": {
            "description": "Train Real + T5 Paraphrases, Test Real",
            "training_records": len(combined_para_records),
            "real_records": len(train_records),
            "synthetic_records": len(paraphrase_records),
            "held_out_test": test_trstr_para,
            "unseen_benchmark": {
                "transformer_only": unseen_trstr_para['modes']['transformer_only'],
                "hybrid": unseen_trstr_para['modes']['hybrid'],
                "generalization_lift": unseen_trstr_para.get('generalization_lift'),
            },
        },
        "TRSTR_LLM": {
            "description": "Train Real + LLM-Direct Gemini Generation, Test Real",
            "training_records": trstr_llm_result['total_records'],
            "real_records": trstr_llm_result['real_count'],
            "synthetic_records": trstr_llm_result['synthetic_count'],
            "mined_negatives": trstr_llm_result['mined_count'],
            "held_out_test": test_trstr_llm,
            "unseen_benchmark": {
                "transformer_only": unseen_trstr_llm['modes']['transformer_only'],
                "hybrid": unseen_trstr_llm['modes']['hybrid'],
                "generalization_lift": unseen_trstr_llm.get('generalization_lift'),
            },
        },
    },
    "evaluation_protocol": {
        "training_hyperparameters": {
            "max_steps": 2500,
            "eval_frequency": 50,
            "patience": 400,
            "backbone": "roberta-base",
        },
        "test_set": "data/training/test.spacy (100% authentic, zero synthetic)",
        "dev_set": "data/training/dev.spacy (100% authentic, zero synthetic)",
        "unseen_benchmark": "data/test/unseen_benchmark.jsonl (65 unseen terms)",
    },
}

report_path = "data/evaluation_report_3way_comparison.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(comparison_report, f, indent=2, ensure_ascii=False)

print(f"Full 3-way comparison report saved to {report_path}")

Full 3-way comparison report saved to data/evaluation_report_3way_comparison.json


---
## Phase 16: Findings & Thesis Methodological Summary

### Comparative Conclusions:

| Condition | Description | Training Pool |
| :--- | :--- | :--- |
| **TRTR** | Baseline — authentic data only | Real training split only |
| **TRSTR-Paraphrase** | T5 paraphrase augmentation of real sentences | Real + T5 paraphrases |
| **TRSTR-LLM** | LLM-direct novel sentence generation (Gemini) | Real + Gemini synthetic |

All 3 conditions were evaluated under **identical hyperparameters** (`max_steps=2500`, `patience=400`, `eval_frequency=50`) on the **exact same real evaluation partitions** (dev/test splits, unseen benchmark).

> **Strict Evaluation Isolation**: Synthetic data exists **strictly on the training side**. All validation and test sets consist solely of authentic records. Synthetic data never touches the evaluation side.

> [!WARNING]
> **Limitations Disclaimer**: Any synthetic augmentation is introduced to compensate for the limited volume of the authentic annotated corpus. TRTR remains the reference point for real-data-only performance. Augmented results should not be misconstrued as evidence that the system was trained exclusively on authentic data.

---
---
# CROSS-VALIDATION
---
## Phase 17: 5-Fold Cross-Validation (TRTR vs. TRSTR-Paraphrase vs. TRSTR-LLM)

To establish statistical significance beyond a single train/dev/test split, this phase runs **Stratified K-Fold Cross-Validation** over all 1,241 authentic records in `data/data.jsonl`.

### Rigorous Protocol:
1. **Pure Real Validation**: In each fold, 20% of authentic records serve as the held-out validation fold. Synthetic data **never** enters the validation fold.
2. **Three Canonical Conditions**:
   - **`TRTR`**: Real training fold only.
   - **`TRSTR-Paraphrase`**: Real training fold + T5 synthetic paraphrases (strictly filtered against validation sentences).
   - **`TRSTR-LLM`**: Real training fold + Gemini LLM synthetic data + curated mined hard negatives (strictly filtered against validation sentences).
3. **Statistical Aggregation**: Reports **Mean ± Standard Deviation** across all folds for Strict F1, Precision, Recall, and Unseen Benchmark Generalization.
4. **Resume-Safe Caching**: Fold checkpoints and per-fold metrics are saved under `data/cv/results/` so the run can be resumed seamlessly.


In [26]:
# Free GPU memory before cross-validation
import gc
import torch
import importlib
import scripts.cross_validation
importlib.reload(scripts.cross_validation)
from scripts.cross_validation import run_3way_cross_validation

for var_name in ["pipe_trtr", "pipe_trstr_para", "pipe_trstr_llm", "pipeline"]:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Execute 3-way K-Fold Cross-Validation
# Note: Per-fold results are cached in data/cv/results/ so execution is resume-safe.
cv_results = run_3way_cross_validation(
    data_path="data/data.jsonl",
    terms_csv_path="data/terms.csv",
    unseen_benchmark_path="data/test/unseen_benchmark.jsonl",
    n_splits=5,
    conditions=["trtr", "trstr_paraphrase", "trstr_llm"],
    max_steps=1500,
    eval_frequency=50,
    patience=250,
    force_train=True
)


[2026-11-07 06:13:19,577] INFO: Loaded 1320 unique authentic documents from data/data.jsonl.
[2026-11-07 06:13:19,580] INFO: Loaded 565 synthetic paraphrases.
[2026-11-07 06:13:19,582] INFO: Loaded 571 LLM synthetic records.
[2026-11-07 06:13:19,583] INFO: Loaded 164 mined hard negatives.
[2026-11-07 06:13:28,090] INFO: Saved 264 docs (293 valid entities, 0 dropped) to data/cv/fold_0/val_real.spacy
[2026-11-07 06:13:28,326] INFO: Saved 1056 docs (1180 valid entities, 1 dropped) to data/cv/fold_0/train_trtr.spacy
[2026-11-07 06:13:28,606] INFO: Saved 1493 docs (1473 valid entities, 1 dropped) to data/cv/fold_0/train_trstr_paraphrase.spacy
[2026-11-07 06:13:28,896] INFO: Saved 1750 docs (1735 valid entities, 1 dropped) to data/cv/fold_0/train_trstr_llm.spacy
[2026-11-07 06:13:37,828] INFO: Saved 264 docs (291 valid entities, 0 dropped) to data/cv/fold_1/val_real.spacy
[2026-11-07 06:13:38,056] INFO: Saved 1056 docs (1182 valid entities, 1 dropped) to data/cv/fold_1/train_trtr.spacy
[2026

ℹ Saving to output directory: models/cv/trtr_f0
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[2026-11-07 06:14:07,090] [INFO] Initialized pipeline components: ['transformer', 'ner']
[2026-11-07 06:14:07,090] INFO: Initialized pipeline components: ['transformer', 'ner']


✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  -------------  --------  ------  ------  ------  ------
  0       0         320.67    545.28    2.89    1.55   22.53    0.03
  2      50       14926.80  25453.48    0.00    0.00    0.00    0.00
  5     100        2957.45   5486.24   29.93   49.22   21.50    0.30
  8     150        3508.36   4482.17   46.90   52.08   42.66    0.47
 11     200        2647.47   3642.09   52.39   59.57   46.76    0.52
 14     250        1968.38   2845.69   59.17   60.00   58.36    0.59
 17     300        1543.98   2402.70   63.33   60.12   66.89    0.63
 19     350        1671.12   2280.62   64.67   66.67   62.80    0.65
 22     400        1150.83   1947.56   69.15   68.69   69.62    0.69
 25     450        1008.33   1779.99   66.02   62.50   69.97    0.66
 28     5

[2026-11-07 06:24:46,593] INFO: Loaded 372 unique terms from data/terms.csv
[2026-11-07 06:24:46,593] INFO: Loading transformer model from 'models/cv/trtr_f0/model-best'...


✔ Saved pipeline to output directory
models/cv/trtr_f0/model-last


... (82 outputs left out)

[2026-11-07 08:42:03,743] INFO: Loaded 372 unique terms from data/terms.csv
[2026-11-07 08:42:03,744] INFO: Loading transformer model from 'models/cv/trstr_llm_f4/model-best'...


✔ Saved pipeline to output directory
models/cv/trstr_llm_f4/model-last


[2026-11-07 08:42:06,724] INFO: Configured standalone Dictionary matcher with 372 patterns.
[2026-11-07 08:42:06,989] INFO: [DICTIONARY OVERRIDE] Span 'office database' (68:83): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 08:42:12,210] INFO: [DICTIONARY OVERRIDE] Span 'system records' (23:37): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 08:42:14,497] INFO: [DICTIONARY OVERRIDE] Span 'Encode Excel' (0:12): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 08:42:16,250] INFO: [DICTIONARY OVERRIDE] Span 'office database' (36:51): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 08:42:16,351] INFO: [DICTIONARY OVERRIDE] Span 'database' (41:49): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 08:42:16,561] INFO: [DICTIONARY OVERRIDE] Span 'student records' (45:60): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 08:42:19,552] INFO: Per-item results: data/eval_results/cv/trstr_llm/unseen_benchmark_transformer_


      K-FOLD CROSS-VALIDATION SUMMARY (MEAN ± STD DEV)
METRIC                               | TRTR (Real-Only) | TRSTR-Paraphrase |        TRSTR-LLM
------------------------------------------------------------------------------------------
Held-Out Validation Overall F1       |  71.71 ± 2.80% |  71.47 ± 0.92% |  67.95 ± 1.41%
Held-Out Validation Overall Precision |  69.17 ± 2.79% |  69.19 ± 1.61% |  66.05 ± 1.74%
Held-Out Validation Overall Recall   |  74.46 ± 3.14% |  73.99 ± 2.10% |  69.99 ± 1.74%
  IT_TERM F1                         |  72.61 ± 3.78% |  72.33 ± 1.45% |  68.82 ± 1.61%
  CLERICAL_TERM F1                   |  70.05 ± 2.39% |  70.08 ± 2.00% |  66.36 ± 1.53%
Unseen Benchmark TRF Recall          |  79.08 ± 5.02% |  77.23 ± 6.56% |  92.00 ± 5.45%
Unseen Benchmark TRF F1              |  50.85 ± 2.31% |  50.58 ± 2.11% |  72.16 ± 4.13%


---
## Phase 18: Separate Mini TRSTR-LLM Cross-Validation

This is an isolated 5-fold CV run for the `trstr_llm_mini` model trained with `config_trf_mini.cfg` (DistilRoBERTa). It evaluates only the TRSTR-LLM condition and uses separate model, metric, and per-item-result directories, so it cannot reuse the full-model CV checkpoints or summaries.

> **Why the previous CV was fast:** CV is resume-safe. When `force_train=False`, existing fold metrics in `data/cv/results/` are loaded immediately and training is skipped. Set `force_train=True` below when a fresh run is required.


In [27]:
# Free GPU memory before the isolated mini-model CV run
import gc
import torch
import importlib
import scripts.cross_validation
importlib.reload(scripts.cross_validation)
from scripts.cross_validation import run_mini_trstr_llm_cross_validation

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

cv_trstr_llm_mini_results = run_mini_trstr_llm_cross_validation(
    data_path="data/data.jsonl",
    terms_csv_path="data/terms.csv",
    unseen_benchmark_path="data/test/unseen_benchmark.jsonl",
    n_splits=5,
    max_steps=1500,
    eval_frequency=50,
    patience=250,
    force_train=True,
)


[2026-11-07 08:42:24,412] INFO: CV folds already exist at data/cv. Loading metadata.
[2026-11-07 08:42:24,414] INFO: Loaded 85 unseen benchmark samples from data/test/unseen_benchmark.jsonl
[2026-11-07 08:42:24,414] INFO: 
STARTING CV EVALUATION: TRSTR_LLM
[2026-11-07 08:42:24,631] INFO: Training TRSTR_LLM (Fold 0) for up to 1500 steps...
[2026-11-07 08:42:24,758] [INFO] Set up nlp object from config
[2026-11-07 08:42:24,758] INFO: Set up nlp object from config
[2026-11-07 08:42:24,773] [INFO] Pipeline: ['transformer', 'ner']
[2026-11-07 08:42:24,773] INFO: Pipeline: ['transformer', 'ner']
[2026-11-07 08:42:24,778] [INFO] Created vocabulary
[2026-11-07 08:42:24,778] INFO: Created vocabulary
[2026-11-07 08:42:24,779] [INFO] Finished initializing nlp object
[2026-11-07 08:42:24,779] INFO: Finished initializing nlp object


ℹ Saving to output directory: models/cv_trstr_llm_mini/trstr_llm_f0
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================


[2026-11-07 08:42:26,572] [INFO] Initialized pipeline components: ['transformer', 'ner']
[2026-11-07 08:42:26,572] INFO: Initialized pipeline components: ['transformer', 'ner']


✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  -------------  --------  ------  ------  ------  ------
  0       0         175.09    282.57    2.98    1.60   22.53    0.03
  1      50       17676.25  29917.22    0.00    0.00    0.00    0.00
  3     100        2698.06   5077.85    0.00    0.00    0.00    0.00
  5     150        3713.95   4903.14   27.27   45.60   19.45    0.27
  7     200        3509.05   4155.66   34.53   45.05   27.99    0.35
  9     250        6171.23   4444.02   35.99   54.11   26.96    0.36
 11     300        1947.04   2992.83   48.39   59.11   40.96    0.48
 12     350        1597.30   2593.10   55.00   57.68   52.56    0.55
 14     400        1516.79   2313.38   56.81   66.06   49.83    0.57
 16     450        1128.50   1941.51   63.78   64.79   62.80    0.64
 18     5

[2026-11-07 08:50:39,352] INFO: Loaded 372 unique terms from data/terms.csv
[2026-11-07 08:50:39,352] INFO: Loading transformer model from 'models/cv_trstr_llm_mini/trstr_llm_f0/model-best'...


✔ Saved pipeline to output directory
models/cv_trstr_llm_mini/trstr_llm_f0/model-last


... (22 outputs left out)

[2026-11-07 09:14:05,935] INFO: Loaded 372 unique terms from data/terms.csv
[2026-11-07 09:14:05,936] INFO: Loading transformer model from 'models/cv_trstr_llm_mini/trstr_llm_f4/model-best'...


✔ Saved pipeline to output directory
models/cv_trstr_llm_mini/trstr_llm_f4/model-last


[2026-11-07 09:14:07,998] INFO: Configured standalone Dictionary matcher with 372 patterns.
[2026-11-07 09:14:08,218] INFO: [DICTIONARY OVERRIDE] Span 'office database' (68:83): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 09:14:09,592] INFO: [DICTIONARY OVERRIDE] Span 'Annual Investment Program (AIP) System' (42:80): Dictionary 'IT_TERM' overrode ML 'CLERICAL_TERM'
[2026-11-07 09:14:12,024] INFO: [DICTIONARY OVERRIDE] Span 'system records' (23:37): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 09:14:12,452] INFO: [DICTIONARY OVERRIDE] Span 'PK Forms' (221:229): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 09:14:12,453] INFO: [DICTIONARY OVERRIDE] Span 'YAKAP Forms' (234:245): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 09:14:12,717] INFO: [DICTIONARY OVERRIDE] Span 'excel digitilization' (68:88): Dictionary 'CLERICAL_TERM' overrode ML 'IT_TERM'
[2026-11-07 09:14:13,003] INFO: [DICTIONARY OVERRIDE] Span 'incident tracking'


      K-FOLD CROSS-VALIDATION SUMMARY (MEAN ± STD DEV)
METRIC                               |        TRSTR-LLM
------------------------------------------------------------------------------------------
Held-Out Validation Overall F1       |  66.77 ± 2.91%
Held-Out Validation Overall Precision |  63.70 ± 2.94%
Held-Out Validation Overall Recall   |  70.18 ± 3.31%
  IT_TERM F1                         |  67.64 ± 2.29%
  CLERICAL_TERM F1                   |  65.20 ± 4.39%
Unseen Benchmark TRF Recall          |  89.85 ± 1.85%
Unseen Benchmark TRF F1              |  70.12 ± 2.68%
